<a href="https://colab.research.google.com/github/Akanshajoshiiii/SpeechLAB/blob/main/Copy_of_Lab6_MFCC_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Load the audio signal

In [ ]:
from scipy.io import wavfile
sample_rate, signal = wavfile.read('BAK.wav')

/tmp/ipykernel_3404/49550083.py:2: WavFileWarning: Chunk (non-data) not understood, skipping it.
  sample_rate, signal = wavfile.read('BAK.wav')


In [ ]:
signal

array([[ 9, 27],
       [ 5, 13],
       [-6, -2],
       ...,
       [ 0,  0],
       [ 0,  0],
       [ 0,  0]], dtype=int16)

In [ ]:
if signal.shape[1] == 2:
    signal = signal.mean(axis=1).astype(signal.dtype)

In [ ]:
signal[1:]

array([  9,  -4, -24, ...,   0,   0,   0], dtype=int16)

In [ ]:
signal[:-1]

array([18,  9, -4, ...,  0,  0,  0], dtype=int16)

In [ ]:
len(signal)

367538

Pre-emphasis

In [ ]:
import numpy as np
pre_emphasized_signal = np.append(signal[0], signal[1:] - 0.97 * signal[:-1])


In [ ]:
pre_emphasized_signal[:10]

array([  9.  ,  27.  ,  -3.73, -13.19, -10.85, -14.61, -17.18, -24.06,
       -10.69, -19.78])

Framing and Windowing

In [ ]:
from scipy.signal import get_window

frame_size = 0.025 # 25 ms
hop_size = 0.01 # 10 ms
frame_length = int(round(sample_rate * frame_size))
hop_length = int(round(sample_rate * hop_size))


In [ ]:
def frame_signal(signal, frame_length, hop_length):
    # Calculate number of frames
    num_frames = int(np.ceil(float(np.abs(len(signal) - frame_length)) / hop_length)) + 1
    # Pad signal to ensure all frames are full
    pad_length = int((num_frames - 1) * hop_length + frame_length)
    zeros = np.zeros((pad_length - len(signal),))
    pad_signal = np.concatenate((signal, zeros))

    # Extract frames using a sliding window
    indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + \
              np.tile(np.arange(0, num_frames * hop_length, hop_length), (frame_length, 1)).T
    frames = pad_signal[indices.astype(np.int32, copy=False)]
    return frames

In [ ]:

frames=frame_signal(pre_emphasized_signal, frame_length, hop_length)

In [ ]:
num_frames = int(np.ceil(float(np.abs(10 - 4)) / 2)) + 1
np.tile(np.arange(0, 4), (num_frames, 1))

array([[0, 1, 2, 3],
       [0, 1, 2, 3],
       [0, 1, 2, 3],
       [0, 1, 2, 3]])

In [ ]:
np.tile(np.arange(0, 4*2, 2), (4, 1)).T

array([[0, 0, 0, 0],
       [2, 2, 2, 2],
       [4, 4, 4, 4],
       [6, 6, 6, 6]])

In [ ]:
# Create a Hamming window of the same length as your frames
window = np.hamming(frame_length)

# Multiply every frame by the window function
windowed_frames = frames * window

Fast Fourier Transform

In [ ]:
from scipy.fftpack import fft
NFFT = 512 # Common FFT size
power_spectrum = (1.0 / NFFT) * (np.abs(fft(windowed_frames, NFFT))**2)

Mel Filterbank Application

In [ ]:
# Functions to convert between frequency and Mel scale:
def hz_to_mel(hz):
    return 2595 * np.log10(1 + hz / 700)
def mel_to_hz(mels):
    return 700 * (10**(mels / 2595) - 1)


In [ ]:
n_filters=26
low_freq_mel = 0
high_freq_mel = (2595 * np.log10(1 + (sample_rate / 2) / 700))  # Convert Hz to Mel
mel_points = np.linspace(low_freq_mel, high_freq_mel, n_filters + 2)
hz_points = (700 * (10**(mel_points / 2595) - 1))  # Convert Mel to Hz
bin = np.floor((NFFT + 1) * hz_points / sample_rate)#converting to dicrete FFT bins

fbank = np.zeros((n_filters, int(np.floor(NFFT/2  + 1))))#intilialising the filter banks
for m in range(1, n_filters + 1):
  f_m_minus = int(bin[m - 1])
  f_m = int(bin[m])
  f_m_plus = int(bin[m + 1])
  for k in range(f_m_minus, f_m):
    fbank[m - 1, k] = (k - bin[m - 1]) / (bin[m] - bin[m - 1])
  for k in range(f_m, f_m_plus):
    fbank[m - 1, k] = (bin[m + 1] - k) / (bin[m + 1] - bin[m])

filter_banks = np.dot(power_spectrum[:, :NFFT // 2 + 1], fbank.T)
filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)  # Numerical stability

Logarithm of Energies

In [ ]:
log_mel_spectrum = 10 * np.log10(filter_banks)

Cepstrum Coefficients

In [ ]:
from scipy.fftpack import dct
num_ceps = 13 # Common number of coefficients to keep
mfccs = dct(log_mel_spectrum, type=2, axis=1, norm='ortho')[:, :num_ceps]